In [2]:
# УСТАНОВКА ВСЕХ НУЖНЫХ БИБЛИОТЕК ДЛЯ БОТА
!pip install -q pytelegrambotapi torch transformers pillow accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 10.1 MB/s eta 0:00:00


In [8]:
# Подключаем все нужные библиотеки
import telebot
import random
import torch
import re
import warnings
from collections import Counter
from PIL import Image
from io import BytesIO
from difflib import SequenceMatcher
from telebot import types
from transformers import BlipProcessor, BlipForConditionalGeneration
from datetime import datetime

# Отключаем лишние предупреждения
warnings.filterwarnings("ignore")
# Устанавливаем точность вычислений для нейросети
torch.set_float32_matmul_precision("medium")

# ===================== НАСТРОЙКИ БОТА =====================
# ID администратора
ADMIN_ID = 6123260381
# Токен телеграм бота
BOT_TOKEN = "8785055458:AAGQWg8mXilRy6aFn_dHDcTPhZG1Owymo_U"
# Лимит логов
MAX_LOGS = 300
# Определяем, использовать видеокарту или процессор
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ===================== ЗАГРУЗКА НЕЙРОСЕТИ BLIP =====================
# Загружаем процессор для обработки фото
processor = BlipProcessor.from_pretrained("Salesforce/BLIP-image-captioning-base")
# Загружаем саму модель для описания картинок
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/BLIP-image-captioning-base",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

# Режим оценки модели (не обучение)
model.eval()

# Параметры генерации описания к фото
GEN_ARGS = {
    "max_new_tokens": 48,       # Максимальная длина описания
    "min_new_tokens": 8,       # Минимальная длина
    "num_beams": 3,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.2,  # Запрет повторений
    "do_sample": False
}

# ===================== ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ =====================
# Проверка на слишком повторяющиеся слова в описании
def is_spam_caption(text: str, repeat_threshold: int = 3) -> bool:
    words = re.findall(r"\b[a-zA-Z]{2,}\b", text.lower())
    if not words: return True
    cnt = Counter(words)
    for word, count in cnt.items():
        if count >= repeat_threshold: return True
    for i in range(len(words)-2):
        if words[i] == words[i+1] == words[i+2]: return True
    return False

# Главная функция: получает фото, возвращает готовое описание с большой буквы
@torch.no_grad()
def generate_caption(image: Image.Image) -> str:
    try:
        inputs = processor(image, return_tensors="pt").to(
            DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32
        )
        out = model.generate(**inputs, **GEN_ARGS)
        caption = processor.decode(out[0], skip_special_tokens=True).strip()
        if is_spam_caption(caption): return ""
        if caption:
            caption = caption.capitalize()
        return caption
    except Exception:
        return ""

# Подсчёт процента совпадения двух текстов (для режима 2)
def similarity(a: str, b: str) -> int:
    return round(SequenceMatcher(None, a.lower(), b.lower()).ratio() * 100)

# Автоматически записываем нового пользователя в базу
def reg_user(msg):
    uid = msg.from_user.id
    if uid not in users:
        users[uid] = {"name": msg.from_user.first_name, "user": msg.from_user.username}

# ===================== БАЗА ГОТОВЫХ АНГЛИЙСКИХ ФРАЗ =====================
# Нужны для неправильных вариантов ответа в режиме 1
phrases = {
    6: {
        "people": [
            "I drink pure water each morning",
            "We walk far through green woods",
            "She reads books in quiet room",
            "He draws sketches on white paper",
            "They cook tasty fresh dinners",
            "You listen to soft calm music",
            "Kids play games in small yard",
            "Friends meet near old stone bridge",
            "Work finishes at late evening hour",
            "Travel gives bright new emotions",
            "I write short daily small notes",
            "We clean house every weekend",
            "She learns new simple languages",
            "He fixes broken household items",
            "They watch stars in dark sky",
            "You bake sweet homemade cookies",
            "Men carry heavy wooden crates",
            "Women grow bright summer flowers",
            "Teachers explain hard school tasks",
            "Students study together closely",
            "I run fast in fresh morning air",
            "We swim slow in clear cold lake",
            "She takes calm lonely walks",
            "He builds small wooden shelters",
            "They share warm happy memories",
            "You care for your close family",
            "Guests arrive at our front door",
            "Workers rest after long shifts",
            "Dancers move to gentle rhythm",
            "Singers sing soft sweet ballads",
            "I buy fresh ripe seasonal fruits",
            "We save money for big dreams",
            "She paints vivid nature pictures",
            "He repairs old rusty tools",
            "They plan short weekend journeys",
            "You collect rare vintage objects",
            "Neighbors chat over low fence",
            "Artists create fresh new concepts",
            "Writers tell real life tales",
            "Cyclists ride wide empty roads",
            "Hikers climb small green hills",
            "I fold clean warm laundry clothes",
            "We sort trash for healthy planet",
            "She hugs beloved dear people",
            "He solves tricky logic puzzles",
            "They give food to needy folks",
            "You master useful new abilities",
            "Guides show local secret places",
            "Players train for big tournaments",
            "Chefs mix healthy fresh products"
        ],
        "animals": [
            "Ants carry crumbs across dry ground",
            "Bats fly high in dark night sky",
            "Deer leap through thick green bushes",
            "Eagles soar over tall rocky peaks",
            "Frogs jump near damp wet marshes",
            "Goats climb steep rough mountain slopes",
            "Hawks hunt tiny field rodents",
            "Insects crawl on hard tree bark",
            "Jellyfish float in calm salt water",
            "Koalas sleep on thick tree branches",
            "Lizards bask in hot warm sunlight",
            "Moles dig deep dark soil tunnels",
            "Otters play in fast cold rivers",
            "Panthers hide in dense dark jungle",
            "Quails run through dry tall grass",
            "Ravens fly over grey stone cliffs",
            "Seals rest on cold wet sea rocks",
            "Tigers roam wild thick forests",
            "Whales dive in deep blue ocean",
            "Zebras run on wide hot savanna",
            "Snails crawl on wet green leaves",
            "Squirrels hide nuts for cold winter",
            "Swan glides on still quiet lake",
            "Turtles crawl on warm sand shores",
            "Wasps build small paper hives",
            "Worms burrow in soft dark earth",
            "Beetles crawl on green tree leaves",
            "Buffalo graze in wet wetlands",
            "Butterflies land on fresh petals",
            "Cranes stand in shallow water",
            "Crabs scuttle on rough seashore",
            "Dolphins swim in large groups",
            "Doves fly to safe warm nests",
            "Elephants walk in big herds",
            "Ferrets sneak in narrow spaces",
            "Finches eat small grass seeds",
            "Foxes hunt in quiet meadows",
            "Geese fly in large sky flocks",
            "Hamsters store food in burrows",
            "Hedgehogs roll in sharp spines",
            "Horses run on open fields",
            "Jaguars stalk prey silently",
            "Kangaroos jump on dry plains",
            "Lemurs play in tall trees",
            "Mice hide in dark small holes",
            "Owls hunt at quiet dark night",
            "Pandas eat fresh green bamboo",
            "Rabbits hop in green clear fields",
            "Sheep graze on soft green grass"
        ]
    },
    7: {
        "people": [
            "I enjoy calm quiet morning moments",
            "We explore unknown distant small towns",
            "She studies science in modern classroom",
            "He creates handmade gifts for loved ones",
            "They celebrate holidays with whole family",
            "You improve your skills day by day",
            "Young children laugh loud in green park",
            "Old folks drink tea on warm porch",
            "Busy workers finish tasks before sunset",
            "Creative artists draw vivid nature pictures",
            "I prepare light breakfast every single morning",
            "We gather wild berries in summer forest",
            "She writes long letters to distant friends",
            "He fixes old broken bikes in backyard",
            "They watch colorful sunsets over calm sea",
            "You grow fresh herbs on small windowsill",
            "Kind strangers help each other in street",
            "Smart learners read books to gain knowledge",
            "Strong athletes train hard for competitions",
            "Calm thinkers meditate in peaceful gardens",
            "I take slow walks to clear busy mind",
            "We take warm photos during short journeys",
            "She knits soft scarves for cold winter days",
            "He learns to play simple musical instruments",
            "They organize clean events for local community",
            "You taste different dishes from world cuisines",
            "Local vendors sell fresh goods every morning",
            "Skillful cooks make healthy homemade dinners",
            "Brave travelers visit far remote countries",
            "Gentle musicians play soft tunes at evening",
            "I save small coins for future big purchases",
            "We plant young trees to protect nature",
            "She collects beautiful stones from river shore",
            "He practices typing fast on old keyboard",
            "They discuss interesting topics at evening table",
            "You watch educational films in free spare time",
            "Urban dwellers enjoy small green city corners",
            "Rural farmers tend crops in open fields",
            "Talented dancers perform shows on open stage",
            "Honest people keep promises to close others",
            "I fold laundry neatly inside wooden cabinet",
            "We listen weather news before long trips",
            "She waters potted plants with fresh cold water",
            "He solves difficult puzzles in quiet evening",
            "They donate unused clothes to needy people",
            "You learn basic rules of safe road traffic",
            "Patient tutors help pupils with hard homework",
            "Careful drivers follow all road regulations",
            "Curious kids ask questions about world around"
        ],
        "animals": [
            "Small ants march steady across dry ground",
            "Nocturnal bats hunt bugs at dark twilight",
            "Wild deer wander free in green woodland",
            "Sharp eagles spot prey from high mountain",
            "Bright frogs croak loud near wet swamp",
            "Mountain goats balance on narrow rock ledges",
            "Swift hawks dive fast to catch small rodents",
            "Tiny insects hide under rough tree foliage",
            "Soft jellyfish drift slow in warm ocean waves",
            "Cute koalas rest tight on thick eucalyptus",
            "Quick lizards absorb heat from hot sunlight",
            "Blind moles dig tunnels through dark soil",
            "Smooth newts hide cool under wet river rocks",
            "Playful otters slide down muddy river banks",
            "Large panthers stalk prey in thick rainforest",
            "Brown quails scurry low through dry field grass",
            "Black ravens soar high above rocky grey cliffs",
            "Fat seals sunbathe on cold northern sea rocks",
            "Fierce tigers patrol vast asian green jungles",
            "Spiky urchins cling hard to rough sea stones",
            "Hungry vultures scan wide dry desert valleys",
            "Huge whales migrate far across blue oceans",
            "Small xenus burrow deep in hot dry sand",
            "Fluffy yaks survive cold high mountain air",
            "Striped zebras graze free on hot african savanna",
            "Slow snails glide gently on damp green leaves",
            "Bright salamanders dwell in moist forest shade",
            "Little shrimp swim quick in shallow tide pools",
            "Busy squirrels gather nuts for cold winter",
            "Graceful swans float calm on quiet still lakes",
            "Tough termites build tall mounds in dry soil",
            "Hard turtles trudge slow along sandy beaches",
            "Angry wasps guard small paper hive nests",
            "Sneaky weasels hunt mice in dark underground",
            "Thin worms aerate dense rich garden soil",
            "Shiny beetles crawl slow on green summer leaves",
            "Colorful blue jays flit between tall oak trees",
            "Heavy buffalo wade deep in muddy wet marshes",
            "Delicate butterflies flutter near bright wildflowers",
            "Tall cranes stand still in shallow fresh wetlands",
            "Hard crabs scuttle fast on rough wet seashores",
            "Long crocodiles lurk quiet beside slow rivers",
            "Friendly dolphins leap high in open ocean waves",
            "Slender dragonflies dart above tall green reeds",
            "White doves return safe to cozy wooden nests",
            "Massive elephants travel slow in big groups",
            "Sly ferrets slip through tiny narrow spaces",
            "Tiny finches peck seeds from dry field stalks"
        ]
    },
    8: {
        "people": [
            "I enjoy drinking coffee in early morning",
            "We like walking through the green forest",
            "She loves reading books in quiet room",
            "He enjoys drawing sketches on white paper",
            "They cook tasty meals for whole family",
            "You listen to calm music every evening",
            "Kids play games in small sunny yard",
            "Friends meet together near old stone bridge",
            "I write small notes for my daily planner",
            "We clean our house every saturday morning",
            "She learns new languages in her free time",
            "He fixes broken items around the house",
            "They watch bright stars in dark night sky",
            "You bake sweet cookies for close friends",
            "Workers rest after long hard working day",
            "Dancers move smoothly to gentle music rhythm",
            "Singers sing soft beautiful sweet ballads",
            "I buy fresh fruits from local market today",
            "We save money for our big future dreams",
            "She paints beautiful pictures of bright nature",
            "Cyclists ride fast along empty wide roads",
            "Hikers climb high up small green forest hills"
        ],
        "animals": [
            "Small ants carry crumbs across dry warm ground",
            "Black bats fly high in dark night summer sky",
            "Brown deer leap through thick green forest bushes",
            "Brave eagles soar over tall rocky mountain peaks",
            "Green frogs jump near damp wet marshy lands",
            "Grey goats climb steep rough high mountain slopes",
            "Strong hawks hunt small tiny field rodents",
            "Small insects crawl hard rough old tree bark",
            "Cute koalas sleep tight on thick tree branches",
            "Small lizards bask in hot warm morning sunlight",
            "Little moles dig deep dark soil underground tunnels",
            "Clever otters play in fast cold river waters",
            "Black panthers hide in dense dark jungle trees",
            "Small quails run through dry tall green grass",
            "Black ravens fly over grey stony cliff tops",
            "Grey seals rest on cold wet sea rocky stones",
            "Orange tigers roam wild thick green forests",
            "Big whales dive deep in dark blue ocean water",
            "Striped zebras run fast on wide hot dry savanna",
            "Small snails crawl slow on wet green leafs",
            "Brown squirrels hide nuts for cold winter days",
            "White swan glides smooth on still quiet lake",
            "Green turtles crawl slow on warm sand shores"
        ]
    },
    9: {
        "people": [
            "I enjoy drinking warm coffee in early morning",
            "We love walking together through green forest",
            "She likes reading good books in quiet room",
            "He likes drawing small sketches on white paper",
            "They cook tasty fresh meals for whole family",
            "You listen to soft calm music every evening",
            "Kids play fun games in small sunny yard",
            "Friends meet together near old stone bridge",
            "I write small daily notes for my planner",
            "We clean our house every saturday morning",
            "She learns new foreign languages in free time",
            "He fixes broken household items around house",
            "They watch bright stars in dark night sky",
            "You bake sweet homemade cookies for friends",
            "Workers rest after long hard working day",
            "Dancers move smoothly to gentle music rhythm",
            "Singers sing soft beautiful sweet ballads",
            "I buy fresh fruits from local market today",
            "We save money for our big future dreams",
            "She paints beautiful pictures of bright nature"
        ],
        "animals": [
            "Small ants carry crumbs across dry warm ground",
            "Black bats fly high in dark night summer sky",
            "Brown deer leap through thick green forest bushes",
            "Brave eagles soar over tall rocky mountain peaks",
            "Green frogs jump near damp wet marshy lands",
            "Grey goats climb steep rough high mountain slopes",
            "Strong hawks hunt small tiny field rodents",
            "Small insects crawl hard rough old tree bark",
            "Cute koalas sleep tight on thick tree branches",
            "Small lizards bask in hot warm morning sunlight",
            "Little moles dig deep dark soil underground tunnels",
            "Clever otters play in fast cold river waters",
            "Black panthers hide in dense dark jungle trees",
            "Small quails run through dry tall green grass",
            "Black ravens fly over grey stony cliff tops",
            "Grey seals rest on cold wet sea rocky stones",
            "Orange tigers roam wild thick green forests",
            "Big whales dive deep in dark blue ocean water",
            "Striped zebras run fast on wide hot dry savanna"
        ]
    },
    10: {
        "people": [
            "I enjoy drinking warm coffee in early morning light",
            "We love walking together through green forest paths",
            "She likes reading good books in quiet cozy room",
            "He likes drawing small sketches on white clean paper",
            "They cook tasty fresh meals for whole big family",
            "You listen to soft calm music every late evening",
            "Kids play fun games in small sunny backyard",
            "Friends meet together near old stone small bridge",
            "I write small daily notes for my personal planner",
            "We clean our house every saturday early morning",
            "She learns new foreign languages in her free time",
            "He fixes broken household items around small house",
            "They watch bright stars in dark night summer sky",
            "You bake sweet homemade cookies for close friends",
            "Workers rest after long hard working busy day",
            "Dancers move smoothly to gentle music soft rhythm",
            "Singers sing soft beautiful sweet romantic ballads"
        ],
        "animals": [
            "Small ants carry crumbs across dry warm summer ground",
            "Black bats fly high in dark night summer sky",
            "Brown deer leap through thick green forest bushes",
            "Brave eagles soar over tall rocky mountain peaks",
            "Green frogs jump near damp wet marshy wet lands",
            "Grey goats climb steep rough high mountain slopes",
            "Strong hawks hunt small tiny field rodents",
            "Small insects crawl hard rough old tree bark",
            "Cute koalas sleep tight on thick tree branches",
            "Small lizards bask in hot warm morning sunlight",
            "Little moles dig deep dark soil underground tunnels",
            "Clever otters play in fast cold river waters",
            "Black panthers hide in dense dark jungle trees",
            "Small quails run through dry tall green grass"
        ]
    }
}

# Собираем все фразы в один список, убираем правильный вариант
def get_all_wrong_phrases(correct_text):
    all_phrases = []
    for ln in [6,7,8,9,10]:
        all_phrases.extend(phrases[ln]["people"])
        all_phrases.extend(phrases[ln]["animals"])
    return [p for p in all_phrases if p != correct_text]

# ===================== ГЛОБАЛЬНЫЕ ПЕРЕМЕННЫЕ =====================
users = {}                  # Список всех пользователей
user_stats = {}             # Личная статистика каждого юзера
global_stats = {"g1": 0, "g1_win": 0, "g2": 0} # Общая статистика бота
user_temp = {}              # Временные данные для каждого чата (режим, этап)

# ===================== КЛАВИАТУРЫ (КНОПКИ) =====================
# Обычное меню для всех пользователей
def main_kb():
    kb = types.ReplyKeyboardMarkup(resize_keyboard=True)
    kb.add("🎮 Режим 1", "📸 Режим 2")
    kb.add("📊 Моя статистика", "🌍 Глобальная статистика")
    return kb

# Клавиатура админ панели
def admin_kb():
    kb = types.ReplyKeyboardMarkup(resize_keyboard=True)
    kb.add("📊 Общая статистика", "👥 Список пользователей")
    kb.add("🔁 Сброс всей статистики", "📤 Выгрузка логов")
    kb.add("🧹 Очистить временные данные", "🔒 Вернуться в меню")
    return kb

# Подключаем бота по токену
bot = telebot.TeleBot(BOT_TOKEN, skip_pending=True)

# ===================== КОМАНДЫ АДМИНА =====================
# Открыть админ панель
@bot.message_handler(func=lambda m: m.from_user.id == ADMIN_ID and m.text == "👨‍💻 Админ панель")
def open_admin(msg):
    bot.send_message(msg.chat.id, "👨‍💻 Админ-панель открыта", reply_markup=admin_kb())

# Вернуться из админки в обычное меню
@bot.message_handler(func=lambda m: m.from_user.id == ADMIN_ID and m.text == "🔒 Вернуться в меню")
def admin_back(msg):
    bot.send_message(msg.chat.id, "🏠 Главное меню", reply_markup=main_kb())

# Показать полную статистику бота
@bot.message_handler(func=lambda m: m.from_user.id == ADMIN_ID and m.text == "📊 Общая статистика")
def admin_full_stats(msg):
    wr1 = round(global_stats["g1_win"] / global_stats["g1"] * 100, 2) if global_stats["g1"] else 0
    text = (
        "📊 Полная статистика бота\n"
        f"• Всего игроков: {len(users)}\n"
        f"• Режим 1 игр: {global_stats['g1']}\n"
        f"• Правильных ответов: {global_stats['g1_win']}\n"
        f"• Средний винрейт: {wr1}%\n"
        f"• Режим 2 фото: {global_stats['g2']}"
    )
    bot.send_message(msg.chat.id, text)

# Вывести список всех людей, которые писали боту
@bot.message_handler(func=lambda m: m.from_user.id == ADMIN_ID and m.text == "👥 Список пользователей")
def admin_users_list(msg):
    if not users:
        bot.send_message(msg.chat.id, "👥 Список пользователей пуст")
        return
    text = "👥 Все пользователи:\n"
    for uid, data in users.items():
        text += f"• ID: {uid} | Имя: {data['name']}\n"
    bot.send_message(msg.chat.id, text[:4000])

# Полностью сбросить всю статистику
@bot.message_handler(func=lambda m: m.from_user.id == ADMIN_ID and m.text == "🔁 Сброс всей статистики")
def admin_reset_stats(msg):
    global global_stats, user_stats
    global_stats = {"g1": 0, "g1_win": 0, "g2": 0}
    user_stats = {}
    bot.send_message(msg.chat.id, "✅ Вся статистика сброшена")

# Выгрузка логов (исправил ошибку с action_log)
@bot.message_handler(func=lambda m: m.from_user.id == ADMIN_ID and m.text == "📤 Выгрузка логов")
def admin_logs(msg):
    bot.send_message(msg.chat.id, "📤 Логовая система в следующем обновлении")

# Очистить временные данные у всех
@bot.message_handler(func=lambda m: m.from_user.id == ADMIN_ID and m.text == "🧹 Очистить временные данные")
def admin_clear_temp(msg):
    global user_temp
    user_temp = {}
    bot.send_message(msg.chat.id, "✅ Временные данные всех пользователей очищены")

# ===================== СТАРТ БОТА =====================
# Команда /start
@bot.message_handler(commands=["start"])
def start(msg):
    reg_user(msg)
    # Если пишет админ — дополнительное сообщение
    if msg.from_user.id == ADMIN_ID:
        bot.send_message(msg.chat.id, "👋 Привет, админ!\nНажми «👨‍💻 Админ панель» для управления", reply_markup=main_kb())
    else:
        bot.send_message(msg.chat.id, "👋 Добро пожаловать", reply_markup=main_kb())

# ===================== РЕЖИМ 1 (ВЫБОР ПРАВИЛЬНОГО ОПИСАНИЯ) =====================
# Запуск режима 1
@bot.message_handler(func=lambda m: m.text == "🎮 Режим 1")
def game1_start(msg):
    reg_user(msg)
    user_temp[msg.chat.id] = {"mode": 1, "stage": "wait_photo"}
    bot.send_message(msg.chat.id, "📤 Отправь фото для задания")

# ===================== РЕЖИМ 2 (СВОЁ ОПИСАНИЕ, БОТ ОТВЕТ СКРЫТ ДО ОТВЕТА) =====================
# Запуск режима 2
@bot.message_handler(func=lambda m: m.text == "📸 Режим 2")
def game2_start(msg):
    reg_user(msg)
    user_temp[msg.chat.id] = {"mode": 2, "hidden_caption": None}
    bot.send_message(msg.chat.id, "📤 Отправь фото — придумай своё описание")

# ===================== ОБРАБОТКА ВСЕХ ФОТОГРАФИЙ =====================
@bot.message_handler(content_types=["photo"])
def handle_all_photos(msg):
    cid = msg.chat.id
    reg_user(msg)
    ut = user_temp.get(cid, {})

    # Если сейчас активен режим 1
    if ut.get("mode") == 1 and ut.get("stage") == "wait_photo":
        try:
            file = bot.get_file(msg.photo[-1].file_id)
            img_bytes = bot.download_file(file.file_path)
            img = Image.open(BytesIO(img_bytes)).convert("RGB")
            correct_cap = generate_caption(img)
            if not correct_cap:
                bot.send_message(cid, "❌ Попробуй другое фото")
                return
            user_temp[cid]["correct"] = correct_cap
            user_temp[cid]["stage"] = "wait_num"
            bot.send_message(cid, "🔢 Введи количество неверных вариантов (1–20)")
        except:
            bot.send_message(cid, "❌ Ошибка")
        return

    # Если сейчас активен режим 2 — сохраняем описание скрыто
    if ut.get("mode") == 2:
        try:
            file = bot.get_file(msg.photo[-1].file_id)
            img_bytes = bot.download_file(file.file_path)
            img = Image.open(BytesIO(img_bytes)).convert("RGB")
            cap = generate_caption(img)
            if not cap:
                bot.send_message(cid, "❌ Не получилось обработать фото")
                return
            user_temp[cid]["hidden_caption"] = cap
            bot.send_message(cid, "✏️ Напиши своё описание этого фото:")
        except:
            bot.send_message(cid, "❌ Ошибка обработки")
        return

# ===================== ОБРАБОТКА ВСЕХ ТЕКСТОВЫХ СООБЩЕНИЙ =====================
@bot.message_handler(func=lambda m: True)
def text_router(msg):
    cid = msg.chat.id
    txt = msg.text.strip()
    ut = user_temp.get(cid, {})

    # Режим 1: пользователь вводит количество неправильных ответов
    if ut.get("mode") == 1 and ut.get("stage") == "wait_num":
        try:
            n = int(txt)
            if not 1 <= n <= 20:
                bot.send_message(cid, "❌ Число 1–20")
                return
            correct = ut["correct"]
            wrong_pool = get_all_wrong_phrases(correct)
            if len(wrong_pool) < n: n = len(wrong_pool)
            wrong_vars = random.sample(wrong_pool, n)
            options = wrong_vars + [correct]
            random.shuffle(options)
            user_temp[cid]["opts"] = options
            user_temp[cid]["stage"] = "answer"
            text_out = "💡 Выбери правильное описание фото:\n"
            for i,opt in enumerate(options,1):
                text_out += f"{i}. {opt}\n"
            bot.send_message(cid, text_out)
        except:
            bot.send_message(cid, "❌ Введи цифру")
        return

    # Режим 1: проверяем, какой вариант выбрал пользователь
    if ut.get("mode") == 1 and ut.get("stage") == "answer":
        try:
            num = int(txt)-1
            opts = ut["opts"]
            chosen = opts[num]
            correct = ut["correct"]
            global_stats["g1"] += 1
            if cid not in user_stats:
                user_stats[cid] = {"w":0,"t":0,"s2":0,"c2":0}
            user_stats[cid]["t"] += 1
            if chosen == correct:
                user_stats[cid]["w"] += 1
                global_stats["g1_win"] += 1
                bot.send_message(cid, "✅ Верно! Отправь новое фото")
            else:
                bot.send_message(cid, f"❌ Неверно\nПравильно: {correct}\nОтправь новое фото")
            user_temp[cid]["stage"] = "wait_photo"
        except:
            bot.send_message(cid, "❌ Введи номер варианта")
        return

    # Режим 2: после своего описания показываем результат и ответ бота
    if ut.get("mode") == 2 and ut.get("hidden_caption") is not None:
        cap = ut["hidden_caption"]
        perc = similarity(cap, txt)
        global_stats["g2"] += 1
        if cid not in user_stats:
            user_stats[cid] = {"w":0,"t":0,"s2":0,"c2":0}
        user_stats[cid]["s2"] += perc
        user_stats[cid]["c2"] += 1
        bot.send_message(
            cid,
            f"📊 Совпадение: {perc}%\n"
            f"🤖 Описание бота: {cap}\n"
            f"👤 Твой вариант: {txt}\n"
            f"Отправь новое фото для следующего задания"
        )
        user_temp[cid]["hidden_caption"] = None
        return

    # Кнопка: моя статистика
    if txt == "📊 Моя статистика":
        s = user_stats.get(cid, {"w":0,"t":0,"s2":0,"c2":0})
        wr = round(s["w"]/s["t"]*100,2) if s["t"] else 0
        avg = round(s["s2"]/s["c2"],2) if s["c2"] else 0
        out = (
            f"📊 Твоя статистика\n"
            f"🎮 Режим 1: {s['w']}/{s['t']} | Винрейт {wr}%\n"
            f"📸 Режим 2: Среднее совпадение {avg}% | Попыток: {s['c2']}"
        )
        bot.send_message(cid, out)
        return

    # Кнопка: глобальная статистика
    if txt == "🌍 Глобальная статистика":
        wr = round(global_stats["g1_win"]/global_stats["g1"]*100,2) if global_stats["g1"] else 0
        out = (
            f"🌍 Общая статистика\n"
            f"🎮 Режим 1: Всего {global_stats['g1']} игр | Винрейт {wr}%\n"
            f"📸 Режим 2: Обработано фото {global_stats['g2']}"
        )
        bot.send_message(cid, out)
        return

# ===================== ЗАПУСК БОТА =====================
if __name__ == "__main__":
    print("✅ Бот запущен | Режим2 скрытое описание + Админ панель с функциями")
    bot.infinity_polling(skip_pending=True, timeout=60)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/BLIP-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

✅ Бот запущен | Режим2 скрытое описание + Админ панель с функциями


2026-04-23 19:12:39,811 (__init__.py:1134 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
ERROR:TeleBot:Infinity polling: polling exited
2026-04-23 19:12:39,815 (__init__.py:1136 MainThread) ERROR - TeleBot: "Break infinity polling"
ERROR:TeleBot:Break infinity polling
